In [1]:
# File dirs to use
outputs_postfix = '_v2'
PATH_READ = f'./../3_llm_generation/outputs{outputs_postfix}'
PATH_WRITE = './../4_response_extraction'

# Define the constants
PROMPT_TEMPLATE = '1prompt_templates_system_stats.json'

EXPECTED_FILES_PER_SAMPLE_SIZE = 120 # Set to 120 for all samples
EXPECTED_PROMPT_RESPONSE_VALUES = 25

# When adding new providers, make sure the provider is added also and present in the api_handler.py script
MODELS = [
    {"name":"claude-3-5-sonnet-20240620", "provider":"anthropic"},
    # {"name":"claude-3-sonnet-20240229", "provider":"anthropic"},   
    # {"name":"gemma:7b", "provider":"ollama"},                      
    # {"name":"gpt-3.5-turbo-0125", "provider":"openai"},             
    # # {"name":"gpt-4-turbo", "provider":"openai"},   # Skipped                 
    # {"name":"gpt-4o", "provider":"openai"},                         
    # {"name":"llama2:13b", "provider":"ollama"},                     
    # {"name":"llama3:8b", "provider":"ollama"},                      
    # {"name":"llama3:70b", "provider":"ollama"},                     
    # {"name":"mistral:7b", "provider":"ollama"},                     
    # {"name":"mixtral:8x22b", "provider":"ollama"},                  
    # {"name":"phi3:medium-128k", "provider":"ollama"},               
    # {"name":"phi3:mini-128k", "provider":"ollama"},                
]

ORG_SAMPLE_FILES = [
    # 'rs_size_5',
    # 'rs_size_10',
    # 'rs_size_25',
    # 'rs_size_50',
    # 'rs_size_100',
    'rs_size_150',
]

PROMPT_SHORT_DICT = {
    # 'cot': 'chain_of_thought',
    # 'sot': 'skeleton_of_thought',
    'sc': 'self_consistency',
    # 'gk': 'generated_knowledge',
    # 'ltm': 'least_to_most',
    # 'cov': 'chain_of_verification',
    # 'sbp': 'step_back_prompting',
    # 'rar': 'rephrase_and_respond',
    # 'em': 'emotion_prompt',
    # 'ds': 'directional_stimuli',
    # 'rcai': 'recursive_criticism_and_improvement',
    # 'rp': 'reverse_prompting',
}

In [2]:
import os
import json
import re
import pandas as pd
from multiprocessing import Pool, TimeoutError
import logging
import sys
import math

# Setup logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

#############################
# Compile the regex pattern #
#############################

### Multi Pattern
# Captures: Starts with [, (, or {, followed by an optional string with at least two letters, then seven comma-separated numerical values, and ends with ], ), or }.
# Example: [ 'abc', 1, 2, 3, 4, 5, 6, 7 ]
multi_pattern = re.compile(r'[\[\(\{]{1}[ \t]*(?:[\"\'\`\(]?[a-zA-Z]{2,}[\"\'\`\)]?){1}(?:[ \t]?\,?[ \t]*[\"\'\`\(]?\d+(?:\.\d*)?(?:[eE][+-]?\d+)?[\"\'\`\)]?){7}[ \t]*[\]\)\}]{1}', re.VERBOSE | re.IGNORECASE)

### List New Pattern
# Captures: Similar to multi_pattern, but the starting and ending delimiters [, (, { and ], ), } are optional.
# Example: [ 'abc', 1, 2, 3, 4, 5, 6, 7 ], or 'abc', 1, 2, 3, 4, 5, 6, 7
list_new_pattern = re.compile(r'[\[\(\{]?[ \t]*(?:[\"\'\`\(]?[a-zA-Z]+[\"\'\`\)]?){1}(?:[ \t]?\,?[ \t]*[\"\'\`\(]?\d+(?:\.\d*[ \t]*)?(?:[eE][+-]?\d+)?[\"\'\`\)]?){7}[ \t]*[\]\)\}]?', re.VERBOSE | re.IGNORECASE)

### Dict Pattern
# Captures: Optionally starts with {, contains a key-value pair where the key is a string followed by : or =, and the value is another string, followed by seven additional key-value pairs with numerical values, optionally ends with }.
# Example: { 'key1': 'value1', 'key2': 1, 'key3': 2, 'key4': 3, 'key5': 4, 'key6': 5, 'key7': 6, 'key8': 7 }
dict_pattern = re.compile(r'\{?[ \t]*(?:[\"\'\`]?(?:[a-zA-Z]+)[\"\'\`]?[ \t]*[\:\=][ \t]*[\"\'\`]?(?:[a-zA-Z]+)[\"\'\`]?(?:\,[ \t]*[\"\'\`]?(?:[a-zA-Z]+)[\"\'\`]?[ \t]*[\:\=][ \t]*(?:\d+(?:\.\d*)?(?:[eE][+-]?\d+)?)){7})[ \t]*\}?', re.VERBOSE | re.IGNORECASE)

### Row List Pattern
# Captures: Optionally starts with b, followed by a string in quotes, and then seven comma-separated numerical values, and optionally ends with \.
# Example: b'abc', 1, 2, 3, 4, 5, 6, 7\
row_list_pattern = re.compile(r'b?[\"\'\`]?(?:[a-zA-Z]+)[\"\'\`]?(?:\,[ \t]*(?:\d+(?:\.\d*)?(?:[eE][+-]?\d+)?)){7}\\?', re.VERBOSE | re.IGNORECASE)

### Row Key Pattern
# Captures: A key-value pair where the key is a string, followed by : or =, and the value is a non-digit string followed by a comma, and then seven key-value pairs with numerical values.
# Example: key: 'value', key2=1, key3=2, key4=3, key5=4, key6=5, key7=6, key8=7
row_key_value_space_pattern = re.compile(r'(?:[a-zA-Z]+[ \t]?[a-zA-Z]*)[ \t]*[\:\=][ \t]*(?:b?[\'\"\`]?\D+[\'\"\`]?\,){1}([ \t]*(?:[a-zA-Z]+[ \t]?[a-zA-Z]*)[ \t]*[\:\=][ \t]*(?:\d+(?:\.\d*)?(?:[eE][+-]?\d+)?[ \t]*\,?)){7}', re.VERBOSE | re.IGNORECASE)

### Dict Equal Val Key Pattern
# Captures: Optionally starts with \text{ or a string in quotes, then a string, and ends with } or a quote, followed by seven numerical values possibly separated by & or |.
# Example: \text{key} 1 & 2 & 3 & 4 & 5 & 6 & 7
dict_equal_val_key_pattern = re.compile(r'[ \t]*(?:(?:\\text{)|(?:b?[\'\"\`]))?([a-zA-Z]+)(?:(?:\})|(?:[\'\"\`]))?[ \t]*(?:[\&\|]?[ \t]*((?:\d+(?:\.\d*)?(?:[eE][+-]?\d+)?)(?:[ \t]*)?)){7}', re.VERBOSE | re.IGNORECASE)

# Define multiple regex patterns
regex_patterns = [
    ('multi_pattern',multi_pattern),
    ('list_new_pattern',list_new_pattern),
    ('dict_pattern',dict_pattern),
    ('row_list_pattern',row_list_pattern),
    ('row_key_value_space_pattern',row_key_value_space_pattern),
    ('dict_equal_val_key_pattern',dict_equal_val_key_pattern),
]

# Initialize a dictionary to count matches for each pattern
regex_usage_counter = {name: 0 for name, _ in regex_patterns}

# Get all the files we need to process, so we can do parallel computation and avoid conflict.
def collect_file_paths():
    tasks = []
    for model in MODELS:
        for prompt_short, prompt_method in PROMPT_SHORT_DICT.items():
            directory_path = f"{PATH_READ}/{model['name'].replace(':','-')}/{prompt_short}/"
            if not os.path.exists(directory_path):
                logging.warning(f"Directory not found: {directory_path}")
                continue

            # Collect files for each sample size
            for sample_size in ORG_SAMPLE_FILES:
                sample_files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if '+'+sample_size+'+' in f]
                
                if len(sample_files) != EXPECTED_FILES_PER_SAMPLE_SIZE:
                    logging.warning(f"Expected {EXPECTED_FILES_PER_SAMPLE_SIZE} files for {sample_size} in {directory_path}, found {len(sample_files)}")
                    continue
                
                tasks.append({
                    'model': model['name'],
                    'prompt_method': prompt_method,
                    'prompt_short':prompt_short,
                    'sample_size': sample_size,
                    'org_folder':sample_files[0].split('+')[2],
                    'files': sample_files,
                    'count': len(sample_files)
                })
    
    return tasks

# def apply_regex_patterns(content):
#     """Apply all regex patterns to the content and return unique cleaned data and match counts."""
#     unique_data = set()
#     # Initialize pattern_counts using the first item in each tuple from regex_patterns
#     pattern_counts = {name: 0 for name, _ in regex_patterns}
#     for key, regex in regex_patterns:
#         matches = regex.findall(content)
#         if matches:
#             pattern_counts[key] += len(matches)
#         for match in matches:
#             if isinstance(match, tuple):
#                 match = ''.join(match)
#             cleaned_match = match.strip("[]{}()").replace('"', '').replace("'", "").split(',')
#             cleaned_data = [x.strip().lower() for x in cleaned_match if x.strip()]

#             if len(cleaned_data) == 8 and cleaned_data[0].isalpha():  # Check expected format and string alpha
#                 try:
#                     # Convert strings to float and check for infinity
#                     formatted_data = [cleaned_data[0]] + [float(x) for x in cleaned_data[1:] if math.isfinite(float(x)) and float(x) >= 0]
#                     # Only add the data if all elements converted to float are finite
#                     if len(formatted_data) == 8:
#                         unique_data.add(tuple(formatted_data))
#                 except ValueError:
#                     continue

#     return list(unique_data), pattern_counts

# def apply_regex_patterns(content):
#     """Apply all regex patterns to the content and return unique cleaned data with positions and match counts."""
#     unique_data = []
#     pattern_counts = {name: 0 for name, _ in regex_patterns}
    
#     for key, regex in regex_patterns:
#         for match in regex.finditer(content):
#             pattern_counts[key] += 1
#             start_pos = match.start()
            
#             if isinstance(match.group(), tuple):
#                 match_str = ''.join(match.group())
#             else:
#                 match_str = match.group()
            
#             cleaned_match = match_str.strip("[]{}()").replace('"', '').replace("'", "").split(',')
#             cleaned_data = [x.strip().lower() for x in cleaned_match if x.strip()]

#             if len(cleaned_data) == 8 and cleaned_data[0].isalpha():
#                 try:
#                     formatted_data = [cleaned_data[0]] + [float(x) for x in cleaned_data[1:] if math.isfinite(float(x)) and float(x) >= 0]
#                     if len(formatted_data) == 8:
#                         unique_data.append((start_pos, tuple(formatted_data)))
#                 except ValueError:
#                     continue

#     # Sort by position and remove duplicates while preserving order
#     unique_data.sort(key=lambda x: x[0])
#     seen = set()
#     unique_data = [(pos, data) for pos, data in unique_data if data not in seen and not seen.add(data)]

#     return unique_data, pattern_counts

def apply_regex_patterns(content):
    """Apply all regex patterns to the content and return unique cleaned data with positions and match counts."""
    all_matches = []
    pattern_counts = {name: 0 for name, _ in regex_patterns}
    
    for key, regex in regex_patterns:
        for match in regex.finditer(content):
            pattern_counts[key] += 1
            start_pos = match.start()
            
            if isinstance(match.group(), tuple):
                match_str = ''.join(match.group())
            else:
                match_str = match.group()
            
            cleaned_match = match_str.strip("[]{}()").replace('"', '').replace("'", "").split(',')
            cleaned_data = [x.strip().lower() for x in cleaned_match if x.strip()]

            if len(cleaned_data) == 8 and cleaned_data[0].isalpha():
                try:
                    formatted_data = [cleaned_data[0]] + [float(x) for x in cleaned_data[1:] if math.isfinite(float(x)) and float(x) >= 0]
                    if len(formatted_data) == 8:
                        all_matches.append((start_pos, tuple(formatted_data)))
                except ValueError:
                    continue

    # Sort all matches by position
    all_matches.sort(key=lambda x: x[0])

    # Remove duplicates while preserving order
    seen = set()
    unique_data = []
    for pos, data in all_matches:
        if data not in seen:
            seen.add(data)
            unique_data.append((pos, data))

    return unique_data, pattern_counts

### This feature is important to avoid more than the n expected rows.
### For example the method RCAI - Recursive Critique and Improvement can produce rows that meet the pattern
### However we are only interested in the last n rows since these correspond to the 'final' version of the improved rows.
# def select_last_n_unique_rows(unique_data, expected_responses):
#     """
#     Select the last n unique rows based on their position in the text if there are more than n.
#     Otherwise, return all unique rows in their original order.
#     """
#     if len(unique_data) > expected_responses:
#         return [data for _, data in unique_data[-expected_responses:]]
#     else:
#         return [data for _, data in unique_data]

### This feature is important to avoid more than the n expected rows.
### For example the method RCAI - Recursive Critique and Improvement can produce rows that meet the pattern
### However we are only interested in the last n rows since these correspond to the 'final' version of the improved rows.
def select_last_n_unique_rows(unique_data, expected_responses):
    """
    Select the last n unique rows based on their position in the text if there are more than n.
    Otherwise, return all unique rows in their original order.
    """
    return [data for _, data in unique_data[-expected_responses:]]

# def process_file(task):
#     """Process a single file and return results, including regex pattern effectiveness."""
#     results = []
#     model_name = task['model']
#     prompt_method = task['prompt_method']
#     prompt_short = task['prompt_short']
#     sample_size = task['sample_size']
#     org_folder = task['org_folder']

#     for file_path in task['files']:
#         try:
#             with open(file_path, 'r') as f:
#                 data = json.load(f)
#             content = data.get('content', '').replace('_', '')
#             extracted_data, regex_counts = apply_regex_patterns(content)
            
#             failed_to_capture = ''
#             if len(extracted_data):
#                 failed_to_capture = content

#             results.append({
#                 'model': model_name,
#                 'prompt_method': prompt_method,
#                 'prompt_template':PROMPT_TEMPLATE,
#                 'prompt_short':prompt_short,
#                 'sample_size': sample_size,
#                 'org_folder':org_folder,
#                 'file_path': file_path,
#                 'extracted_rows': extracted_data,
#                 'extracted_data_len': len(extracted_data),
#                 'failed_to_capture':failed_to_capture,
#                 'regex_counts': regex_counts
#             })

#             logging.info(f"Processed file {file_path} with regex counts: {regex_counts}")
#             logging.info(f"Extracted Data ({len(extracted_data)}: {extracted_data})")
#             logging.info(f"Regex Counts: {regex_counts}")
#         except Exception as e:
#             logging.error(f"Error processing file {file_path}: {e}")
#     return results

# def process_file(task):
#     """Process a single file and return results, including regex pattern effectiveness."""
#     results = []
#     model_name = task['model']
#     prompt_method = task['prompt_method']
#     prompt_short = task['prompt_short']
#     sample_size = task['sample_size']
#     org_folder = task['org_folder']

#     for file_path in task['files']:
#         try:
#             with open(file_path, 'r') as f:
#                 data = json.load(f)
#             content = data.get('content', '').replace('_', '')
#             unique_data_with_pos, regex_counts = apply_regex_patterns(content)
            
#             # Select the last 25 unique rows
#             extracted_data = select_last_n_unique_rows(unique_data_with_pos,EXPECTED_PROMPT_RESPONSE_VALUES)
            
#             failed_to_capture = ''
#             if not extracted_data:
#                 failed_to_capture = content

#             results.append({
#                 'model': model_name,
#                 'prompt_method': prompt_method,
#                 'prompt_template': PROMPT_TEMPLATE,
#                 'prompt_short': prompt_short,
#                 'sample_size': sample_size,
#                 'org_folder': org_folder,
#                 'file_path': file_path,
#                 'extracted_rows': extracted_data,
#                 'extracted_data_len': len(extracted_data),
#                 'failed_to_capture': failed_to_capture,
#                 'regex_counts': regex_counts
#             })

#             logging.info(f"Processed file {file_path} with regex counts: {regex_counts}")
#             logging.info(f"Extracted Data ({len(extracted_data)}): {extracted_data}")
#             logging.info(f"Regex Counts: {regex_counts}")
#         except Exception as e:
#             logging.error(f"Error processing file {file_path}: {e}")
#     return results

def process_file(task):
    """Process a single file and return results, including regex pattern effectiveness."""
    results = []
    model_name = task['model']
    prompt_method = task['prompt_method']
    prompt_short = task['prompt_short']
    sample_size = task['sample_size']
    org_folder = task['org_folder']

    for file_path in task['files']:
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            content = data.get('content', '').replace('_', '')
            unique_data_with_pos, regex_counts = apply_regex_patterns(content)
            
            # Select the last 25 unique rows or all if less than 25
            extracted_data = select_last_n_unique_rows(unique_data_with_pos, EXPECTED_PROMPT_RESPONSE_VALUES)
            
            failed_to_capture = ''
            if not extracted_data:
                failed_to_capture = content

            results.append({
                'model': model_name,
                'prompt_method': prompt_method,
                'prompt_template': PROMPT_TEMPLATE,
                'prompt_short': prompt_short,
                'sample_size': sample_size,
                'org_folder': org_folder,
                'file_path': file_path,
                'extracted_rows': extracted_data,
                'extracted_data_len': len(extracted_data),
                'failed_to_capture': failed_to_capture,
                'regex_counts': regex_counts
            })

            logging.info(f"Processed file {file_path} with regex counts: {regex_counts}")
            logging.info(f"Extracted Data ({len(extracted_data)}): {extracted_data}")
            logging.info(f"Regex Counts: {regex_counts}")
        except Exception as e:
            logging.error(f"Error processing file {file_path}: {e}")
    return results

def main():
    tasks = collect_file_paths()
    with Pool(min(len(tasks), os.cpu_count())) as pool:
        async_results = []
        for task in tasks:
            async_result = pool.apply_async(process_file, (task,))
            async_results.append(async_result)

        results = []
        for async_result in async_results:
            try:
                result = async_result.get(timeout=60)  # 60 seconds timeout
                results.extend(result)
            except TimeoutError:
                logging.warning("A file processing operation timed out and was skipped.")

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        print(results_df.head())

        # Save json file including in different format
        results_df.to_json(f'results_dataframe{outputs_postfix}.json',indent=4)
        
        ################################
        ## Dataframe just sample rows ##

        # Load JSON data into a dictionary
        with open(f'results_dataframe{outputs_postfix}.json', 'r') as file:
            data = json.load(file)

        # Convert dictionary to DataFrame
        dfx = pd.DataFrame(data)
        dfx = dfx.drop(columns=['prompt_template','org_folder','file_path','regex_counts','failed_to_capture','extracted_data_len'])

        features = ['target_material','target_thickness','pulse_width','energy','spot_size','intensity','power','cutoff_energy']

        # Explode the 'extracted_rows' column to create a new row for each entry
        dfx = dfx.explode('extracted_rows')

        # Filter out rows where 'extracted_rows' is NaN
        dfx = dfx[dfx['extracted_rows'].notna()]

        # Convert list in 'extracted_rows' to separate columns
        if not dfx.empty and isinstance(dfx.iloc[0]['extracted_rows'], list):
            dfx[features] = pd.DataFrame(dfx['extracted_rows'].tolist(), index=dfx.index)

        # Replace all whitespace in the 'model' column
        dfx['model'] = dfx['model'].str.replace(r'\s+', '', regex=True)

        # Drop the original 'extracted_rows' column
        dfx = dfx.drop(columns=['extracted_rows'])

        # Reset the index
        dfx = dfx.reset_index(drop=True)

        # Make sure all numeric features are floats / numeric
        dfx[features[1:]] = dfx[features[1:]].astype(float)

        dfx.to_csv(f'synthetic_data_rows{outputs_postfix}.csv')
    else:
        logging.info("No data extracted or DataFrame is empty.")

if __name__ == '__main__':
    main()

2025-01-23 12:11:25,813 - INFO - Processed file ./../3_llm_generation/outputs_v2/claude-3-5-sonnet-20240620/sc/claude-3-5-sonnet-20240620+1prompt+d_clean_remove_small_samples_ipr+rs_size_150+sc+2025-01-23T11-50-43.json with regex counts: {'multi_pattern': 25, 'list_new_pattern': 25, 'dict_pattern': 0, 'row_list_pattern': 25, 'row_key_value_space_pattern': 0, 'dict_equal_val_key_pattern': 0}
2025-01-23 12:11:25,815 - INFO - Extracted Data (25): [('plastic', 0.589, 30.0, 2.381, 3.3, 6.435e+20, 79370000000000.0, 5.8), ('plastic', 0.273, 30.0, 2.315, 3.3, 6.257e+20, 77170000000000.0, 4.2), ('plastic', 0.762, 279.0, 1.588, 3.3, 4.615e+19, 5692000000000.0, 4.7), ('plastic', 0.501, 80.0, 2.412, 3.3, 2.445e+20, 30150000000000.0, 5.2), ('plastic', 0.937, 187.0, 2.372, 3.3, 1.029e+20, 12690000000000.0, 6.5), ('plastic', 0.685, 30.0, 2.398, 3.3, 6.481e+20, 79940000000000.0, 6.0), ('plastic', 0.322, 30.0, 2.342, 3.3, 6.329e+20, 78060000000000.0, 4.6), ('plastic', 0.769, 463.0, 2.401, 3.3, 4.205e+1